## In Class Assignment April 7th

In [1]:
#install sweetviz
! pip install sweetviz


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
#imports
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression


In [3]:
#read in data and display first five rows
insurance = pd.read_csv("/Users/donyabehroozi/Downloads/insurance.csv")

insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
#initial EDA with SweetViz
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Trends and Patterns
- The outcome variable, charges, is skewed to the right with a median of $9,382.
- The data shows that most individuals are in their 20s, split between somewhat equally male and female, have an average BMI of 30.7, own 0 children, are non-smokers, and primarily live in the southeast.
- Smoking status appears to provide a great deal of information to insurance charges.
- Region appears to provide a moderate amount of information to regarding one's BMI.
- There is a moderate positive associate between age and insurance charges.

In [6]:
#encode categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [7]:
#prepare data for modeling
X = insurance_encoded.drop('charges', axis = 1)
y = insurance_encoded['charges']

#split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
#scaling features (don't need for RF, but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test) #do not fit_transform on X_test, because we use the mean/std dev from X_train to transform the test values

In [11]:
#baseline linear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#baseline random forest model
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

#print baseline results
print(f"Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}")
print(f"Baseline Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}")

Baseline Linear Regression MSE: 33566439.74, R2: 0.78
Baseline Random Forest MSE: 22179121.67, R2: 0.86


Both models perform well on the test data in terms of R^2. The baseline random forest model performs better than the baseline linear regression model due to having a smaller MSE and larger R^2. 86% of the variance in insurance charges is explained by the random forest model, while 78% of variance in the insurance charges is explained by the linear regression model.

In [16]:
#CV with baseline models
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv = 5, scoring = 'r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv = 5, scoring = 'r2')

print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr.mean()}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf.mean()}')

Baseline Linear Regression CV R2 Scores: 0.7330691648735672
Baseline Random Forest CV R2 Scores: 0.8229556792392696


With cross validation, we can see that on average, the baseline random forest model still performs better than the linear regression model on the test data due to having the larger R^2.

In [20]:
#gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

search = GridSearchCV(baseline_rf, 
                      param_grid, 
                      cv = 5, 
                      scoring = 'r2', 
                      n_jobs = -1)
                     
#fit bagging tree model training data and obtain best tree based on cross validated accuracy
RF_tree = search.fit(X_train, y_train) 

In [ ]:
#diplay best estimators
RF_tree.best_estimator_ 

RandomForestRegressor(max_depth=10, max_features='log2', min_samples_split=5,
                      n_estimators=500, random_state=42)

The best set of hyperparameters are:
- max_depth = 10
- max_features = log2
- min_samples_split = 5
- n_estimators = 500

In [ ]:
#display top 20 models
pd.DataFrame(RF_tree.cv_results_).sort_values('rank_test_score')[['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 'param_max_features', 'mean_test_score', 'rank_test_score']].head(20) 

,param_n_estimators,param_max_depth,param_min_samples_split,param_max_features,mean_test_score,rank_test_score
41,500,10,5,log2,0.841838,1
68,500,20,5,log2,0.841282,2
95,500,30,5,log2,0.841282,2
14,500,None,5,log2,0.841282,2
40,200,10,5,log2,0.841202,5
38,500,10,2,log2,0.841076,6
44,500,10,10,log2,0.840716,7
67,200,20,5,log2,0.840662,8
94,200,30,5,log2,0.840662,8
13,200,None,5,log2,0.840662,8


I would use the following set of hyperparameters:

- max_depth = 10
- max_features = log2
- min_samples_split = 5
- n_estimators = 200

This set of hyperparameters appears in row ID 40 within the top 5 best models. This set of hyperparameters results in a very similar performance in terms of average R^2 to the top performing random forest model (84.12% of variation in insurance charges explained in this model vs. 84.18% variation in insurance charges explained by the #1 ranked model) with a low to moderate computational expense. I would avoid choosing a set of hyperparameters where n_estimators is 500, as this will cost us much more time than a random forest model with n_estimators = 100 or 200. 

Realistically though, all of these models perform very similarly in terms of average R^2. If we wanted, we can even choose a combination of hyperparameters near the bottom of the list, like row ID 36 to make the computational time less costly for us. 

In [23]:
#fit random forest model with best performing set of hyperparameters
RF_best_model = RandomForestRegressor(max_depth=10, max_features='log2', min_samples_split=5, n_estimators=500, random_state=42)

RF_best_model.fit(X_train, y_train)

y_train_pred_rf = RF_best_model.predict(X_train)
y_test_pred_rf = RF_best_model.predict(X_test)

#measure performance
mse_train = mean_squared_error(y_train, y_train_pred_rf)
mse_test = mean_squared_error(y_test, y_test_pred_rf)

r2_train = r2_score(y_train, y_train_pred_rf)
r2_test = r2_score(y_test, y_test_pred_rf)

#print results
print(f"Train RF MSE: {mse_train:.2f}, R2: {r2_train:.2f}")
print(f"Test RF MSE: {mse_test:.2f}, R2: {r2_test:.2f}")

Train RF MSE: 9944478.45, R2: 0.93
Test RF MSE: 20578056.27, R2: 0.87


Yes, the model is overfitting because the MSE is much lower and R^2 is higher for the training data than the test data. The model is performing well on the training set but noticeably worse on the test set (especially in terms of MSE).